In [24]:
from z3 import *
from utils import * 
import sympy as sp
import numpy as np
from itertools import product
from fractions import Fraction
import random

In [39]:
def create_variables(edge_list):
    return {(u, v): sp.Symbol(f"m_{u}_{v}") for u, v in edge_list}


def get_M_matrix(edge_list, n):
    vars = create_variables(edge_list)
    M = sp.zeros(n)

    for (u, v), var in vars.items():
        M[u-1, v-1] = var

    return M, list(vars.values())

def get_L_matrix(edge_list, n, low=-1.0, high=1.0):
    L = np.zeros((n, n))

    for u, v in edge_list:
        L[u-1, v-1] = np.random.uniform(low, high)

    return L

def get_J(B2, n):
    B2 = set(B2)
    return [(i, j) for i in range(1, n+1)
                   for j in range(i+1, n+1)
                   if (i, j) not in B2]

def get_K(B1, n):
    symmetric_B1 = set(B1) | {(j, i) for i, j in B1}
    return [(i, j)
            for i in range(1, n+1)
            for j in range(1, n+1)
            if i == j or (i, j) in symmetric_B1]

def map_pairs(JK):
    return [((a, c), (b, d)) for ((a, b), (c, d)) in JK]

def filter_pairs(JK, A):
    return [((a, b), (c, d)) 
            for ((a, b), (c, d)) in JK
            if A[a-1, b-1] != 0 and A[c-1, d-1] != 0]

def create_equations(JK, A):
    return [
        A[a-1, b-1] * A[c-1, d-1]
        for ((a, b), (c, d)) in JK
    ]

def get_both_linear_equations(JK, A):
    return [
        [A[a-1, b-1] , A[c-1, d-1]]
        for ((a, b), (c, d)) in JK
    ]

In [40]:
# Number of Vertices
n = 3

# Generate DAGs
G1 = nx.DiGraph()
G2 = nx.DiGraph()
V = range(1,n+1)
G1.add_nodes_from(V)
G2.add_nodes_from(V)

# Edge Sets
E1 = [(1,2),(2,3)]
E2 = [(1,2),(1,3)]
B1 = [(2,3)]
B2 = [(2,3)]

# Add Edges
G1.add_edges_from(E1)
G2.add_edges_from(E2)

In [78]:
m = {e: Real(f"m_{e[0]}_{e[1]}") for e in E2}
m

{(1, 2): m_1_2, (1, 3): m_1_3}

In [41]:
L = get_L_matrix(E1, n)
M, vars = get_M_matrix(E2, n)
L, M, vars

(array([[ 0.        , -0.60973247,  0.        ],
        [ 0.        ,  0.        ,  0.25861266],
        [ 0.        ,  0.        ,  0.        ]]),
 Matrix([
 [0, m_1_2, m_1_3],
 [0,     0,     0],
 [0,     0,     0]]),
 [m_1_2, m_1_3])

In [42]:
I = np.eye(n)
A = (I-M).T @ np.linalg.inv(I - L).T
J = get_J(B2,n)
K = get_K(B1,n)
JK = map_pairs(list(product(J,K)))
JK_reduced = filter_pairs(JK, A)
JK_reduced

[((1, 1), (2, 1)), ((1, 1), (3, 1))]

In [67]:
eq = get_both_linear_equations(JK_reduced, A)
eq[1][1]

-1.0*m_1_3 - 0.157684532378241

In [72]:
expr = eq[0][1]

symbols = expr.free_symbols

z3_vars = {
    str(s): Real(str(s))
    for s in symbols
}

print(z3_vars)

{'m_1_2': m_1_2}


In [73]:
def sympy_to_z3(expr, z3_vars):
    if expr.is_Number:
        return RealVal(str(expr))

    elif expr.is_Symbol:
        return z3_vars[str(expr)]

    elif expr.is_Add:
        return sum(sympy_to_z3(arg, z3_vars) for arg in expr.args)

    elif expr.is_Mul:
        result = RealVal("1")
        for arg in expr.args:
            result *= sympy_to_z3(arg, z3_vars)
        return result

    else:
        raise ValueError(f"Unsupported: {expr}")

In [74]:
z3_expr = sympy_to_z3(eq[0][1], z3_vars)

print(z3_expr)

0 + -304866233046007/500000000000000 + 1*-1*m_1_2


In [25]:
n = 20
k = 30

x = [Real(f"x{i}") for i in range(n)]

s = Solver()

for i in range(k):
    f = sum(Q(random.randint(-10000,10000),random.randint(-10000,10000))*x[j] for j in range(n)) + Q(random.randint(-10000,10000),random.randint(-10000,10000))
    g = sum(Q(random.randint(-10000,10000),random.randint(-10000,10000)) for j in range(n)) + Q(random.randint(-10000,10000),random.randint(-10000,10000))

    b = Bool(f"choose_{i}")

    s.add(If(b, f == 0, g == 0))

print(s.check())

unsat


In [35]:
n = 3
k = 30

x = [Real(f"x{i}") for i in range(n)]

s = Solver()

for i in range(k):
    f = sum(Fraction(random.randint(-10000,10000),random.randint(-10000,10000))*x[j] for j in range(n)) + Fraction(random.randint(-10000,10000),random.randint(-10000,10000))
    g = sum(Fraction(random.randint(-10000,10000),random.randint(-10000,10000)) for j in range(n)) + Fraction(random.randint(-10000,10000),random.randint(-10000,10000))

    b = Bool(f"choose_{i}")

    s.add(If(b, f == 0, g == 0))

print(s.check())

unsat


[((1, 1), (2, 1)), ((1, 1), (3, 1))]

[((1, 1), (2, 1)),
 ((1, 2), (2, 2)),
 ((1, 2), (2, 3)),
 ((1, 3), (2, 2)),
 ((1, 3), (2, 3)),
 ((1, 1), (3, 1)),
 ((1, 2), (3, 2)),
 ((1, 2), (3, 3)),
 ((1, 3), (3, 2)),
 ((1, 3), (3, 3))]